In [6]:
import torch

from simpleMB import SimpleMB

import numpy as np
import matplotlib.pyplot as plt

class Args():
    pass

args = Args()
args.device = "cpu"

In [10]:
potential = SimpleMB(args,n_in=2)

x0, xf = potential.initial_point.detach(), potential.final_point.detach()

potential.force_func(x1)

(tensor(0), tensor([-0.5904,  0.7353]))

In [11]:
def S2_action(path):
    #print(path)
    result = 0.0
    for i in range(path.shape[0]-1):
        first_term = torch.square((path[i+1,:] - path[i,:])) * (xi/4/ dt)
        second_term = torch.square(potential.force_func(path[i,:])[1]) * (dt/4/xi)
        third_term = potential.laplace(path[i,:]) * (dt* D / torch.tensor(2.0))
        result = result + torch.sum(first_term + second_term + third_term)
    
    return result

def simple_action(path):
    #print(path)
    result = 0.0
    for i in range(path.shape[0]-1):
        first_term = torch.square((path[i+1,:] - path[i,:])) * (xi/ dt)
        f_n = potential.force_func(path[i,:])[1]
        f_np = potential.force_func(path[i+1,:])[1]
        second_term = (torch.square(f_n) + torch.square(f_np)) * (dt/xi/2.0)
        third_term = (path[i+1,:] - path[i,:]) * (f_np - f_n)
        result = result + torch.sum(first_term + second_term + third_term)
    
    return result / torch.tensor(4.0)

In [ ]:
line_density = 30

line_x = torch.linspace(x0[1], xf[1],line_density)
line_y = torch.linspace(x0[0], xf[0], line_density)
line_points = torch.stack((line_x, line_y), axis=-1)

dt = 0.1
iterations = 1000
alpha = 2e-1
write_every = 100

xi = torch.tensor(0.1)
D = torch.tensor(10)

optimizer = torch.optim.Adam([line_points], lr = alpha)

gif_data = []

#Choose action
#action_func = simple_action
action_func = S2_action

for i in range(iterations):
    line_points.requires_grad = True
    #print(line_points, torch.flip(line_points, dims = (0,)))
    action = action_func(line_points)
    reverse_action = action_func(torch.flip(line_points, dims = (0,)))

    # It seems likely that they are the same. It probably can be proven
    total_action = action + reverse_action

    optimizer.zero_grad()

    grads, = torch.autograd.grad(total_action, line_points)
    
    

    with torch.no_grad():
        #grads = grads / torch.sum(torch.abs(grads) + 1e-8)
        grads[0,:], grads[-1,:] = torch.zeros(2), torch.zeros(2)
        #print(line_points)
        #print(line_points.shape, grads.shape)
        line_points.grad = grads
       # optimizer.apply_gradients(())
        optimizer.step()

        draw_points = line_points.detach()
        #line_points = (line_points - alpha * grads).detach()
        #print(line_points)

    if i % write_every == 0:
        print(f"Total action for the step {i} is {(action+reverse_action).detach().numpy()}")
        num_points = 100
        line_density = 10
        x_values = torch.linspace(potential.Lx, potential.Hx , num_points)
        y_values = torch.linspace(potential.Ly, potential.Hy, num_points)

        x, y = torch.meshgrid(x_values, y_values)
        z = potential.U_split(x, y)

        fig, ax = plt.subplots()
        #contour_plot = ax.contour(x, y, z, levels=[potential.U_min, potential.U_max], cmap='viridis')
        colorbar = ax.imshow(z, extent=(x_values.min(), x_values.max(), y_values.min(), y_values.max()), vmin=potential.U_min, vmax=potential.U_max, origin='lower', cmap='viridis', aspect='auto')
        ax.set(xlabel="x-axis", ylabel="y-axis", title="Contour Plot")
        plt.colorbar(colorbar)

        scatter_plot = ax.scatter(draw_points[:,0], draw_points[:,1], s=1, c='red', label='Line')
        quiver_plot = ax.quiver(draw_points[:,0], draw_points[:,1], -grads[:,0], -grads[:,1], scale_units='xy', angles='xy', color='blue', alpha=0.7)
        ax.legend()

        plt.show()

        gif_data.append(draw_points.detach().clone())
